# Phase 1 V2 — Notebook 02: FFIEC Variable Inventory and Code Mapping

**Purpose.** Inventory FFIEC variables by year and quarter; verify the actual reporting calendar from file contents; build historical availability and RCFD/RCON equivalence tables; screen for discontinued or replacement codes, Basel-era capital-reporting breaks and FFIEC 051 structural non-reporting; and document the definition, unit and sign treatment of every candidate predictor.

**Dependency.** Run Notebook 01 V2 first. This notebook imports the shared V2 `config.py` and writes only under `Data Final/Version 2 Main Analysis`.

**Important interpretation.** Automatically detected code changes, equivalences and structural breaks are audit screens. Notebook 03 must apply only conceptually verified mappings during extraction and harmonisation.


## 1. Mount Google Drive and import packages


In [ ]:
# Mount Google Drive and import packages

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import csv
import importlib.util
import json
import re
import sys

import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable=None, *args, **kwargs):
        return iterable

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)
print("Google Drive mounted successfully.")


## 2. Import the authoritative V2 configuration and verify Notebook 01


In [ ]:
# Import the authoritative V2 configuration and verify Notebook 01

PROJECT_ROOT = Path("/content/drive/MyDrive/Data Final")
CONFIG_PATH = PROJECT_ROOT / "Version 2 Main Analysis" / "config.py"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"V2 config.py not found: {CONFIG_PATH}\nRun Notebook 01 V2 first."
    )

spec = importlib.util.spec_from_file_location("v2_config", CONFIG_PATH)
v2_config = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = v2_config
spec.loader.exec_module(v2_config)

VERSION_2_MAIN_TIMING_POLICY = v2_config.VERSION_2_MAIN_TIMING_POLICY
EXPECTED_POLICY_KEYS = v2_config.EXPECTED_VERSION_2_MAIN_TIMING_POLICY_KEYS
FFIEC_RAW_DIR = v2_config.FFIEC_RAW_DIR
V2_ROOT = v2_config.V2_ROOT
V2_OUTPUT_ROOT = v2_config.V2_OUTPUT_ROOT
DATA_SCOPE = v2_config.DATA_SCOPE

NB01_OUTPUT_DIR = V2_OUTPUT_ROOT / "01_Raw_Data_Audit"
NB01_AUDIT_PATH = NB01_OUTPUT_DIR / "audit_report_v2.json"
NB02_OUTPUT_DIR = V2_OUTPUT_ROOT / "02_FFIEC_Variable_Inventory"

assert tuple(VERSION_2_MAIN_TIMING_POLICY) == EXPECTED_POLICY_KEYS
assert VERSION_2_MAIN_TIMING_POLICY["analysis_version"] == "v2_bank_lag1_main"
assert VERSION_2_MAIN_TIMING_POLICY["archived_v1_read_only"] is True
assert NB02_OUTPUT_DIR.is_relative_to(V2_ROOT)

required_paths = (FFIEC_RAW_DIR, CONFIG_PATH, NB01_AUDIT_PATH)
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Required Notebook 01 files or FFIEC folder not found:\n"
        + "\n".join(missing_paths)
    )

with NB01_AUDIT_PATH.open("r", encoding="utf-8") as file:
    nb01_audit = json.load(file)

if nb01_audit.get("overall_status") != "PASS":
    raise ValueError("Notebook 01 V2 did not finish with PASS status.")
if nb01_audit.get("policy_keys_exact") is not True:
    raise ValueError("Notebook 01 did not confirm the exact V2 policy key set.")

NB02_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Notebook 01 V2 status: PASS")
print(f"Analysis version: {VERSION_2_MAIN_TIMING_POLICY['analysis_version']}")
print(f"FFIEC folder: {FFIEC_RAW_DIR}")
print(f"Notebook 02 V2 output: {NB02_OUTPUT_DIR}")


## 3. Discover supported FFIEC raw text files


In [ ]:
# Discover supported FFIEC raw text files

SUPPORTED_EXTENSIONS = {".txt", ".csv", ".tsv", ".dat"}

all_ffiec_files = sorted(
    path
    for path in FFIEC_RAW_DIR.rglob("*")
    if path.is_file()
    and not any(part.startswith(".") for part in path.relative_to(FFIEC_RAW_DIR).parts)
)
ffiec_files = [
    path for path in all_ffiec_files
    if path.suffix.lower() in SUPPORTED_EXTENSIONS
]
unsupported_ffiec_files = [
    path for path in all_ffiec_files
    if path.suffix.lower() not in SUPPORTED_EXTENSIONS
]

if not ffiec_files:
    raise ValueError("No supported FFIEC raw text files were found.")

print(f"All FFIEC files: {len(all_ffiec_files):,}")
print(f"Supported raw text files to scan: {len(ffiec_files):,}")
print(f"Other files recorded but not parsed here: {len(unsupported_ffiec_files):,}")
for path in ffiec_files[:10]:
    print(path.relative_to(FFIEC_RAW_DIR))
if len(ffiec_files) > 10:
    print(f"...and {len(ffiec_files) - 10:,} more supported files.")


## 4. Define robust FFIEC header, calendar and metadata helpers


In [ ]:
# Define robust FFIEC header, calendar and metadata helpers

FINANCIAL_CODE_PATTERN = re.compile(r"^[A-Z]{4}[A-Z0-9]{4}$")

DATE_ALIASES = (
    "Reporting Period End Date",
    "REPORTING PERIOD END DATE",
    "Report Date",
    "REPORT_DATE",
)
FORM_ALIASES = (
    "Financial Institution Filing Type",
    "FINANCIAL INSTITUTION FILING TYPE",
    "Filing Type",
    "FORM_TYPE",
)


def detect_encoding(path):
    for encoding in ("utf-8-sig", "utf-8", "latin-1"):
        try:
            with path.open("r", encoding=encoding, errors="strict") as file:
                file.readline()
            return encoding
        except UnicodeDecodeError:
            continue
    raise UnicodeError(f"Unable to detect encoding for {path.name}")


def detect_delimiter(path, encoding):
    with path.open("r", encoding=encoding, errors="replace") as file:
        first_line = file.readline()
    delimiter_counts = {
        "\t": first_line.count("\t"),
        ",": first_line.count(","),
        "|": first_line.count("|"),
        ";": first_line.count(";"),
    }
    delimiter = max(delimiter_counts, key=delimiter_counts.get)
    if delimiter_counts[delimiter] == 0:
        raise ValueError(f"Delimiter could not be detected for {path.name}")
    return delimiter


def extract_file_year(path):
    text = str(path.relative_to(FFIEC_RAW_DIR))
    years = re.findall(r"(?<!\d)(20\d{2})(?!\d)", text)
    valid_years = [int(year) for year in years if 2001 <= int(year) <= 2026]
    return valid_years[-1] if valid_years else np.nan


def normalise_description(value):
    value = str(value).upper().strip()
    value = re.sub(r"[^A-Z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


def infer_raw_unit(description):
    text = normalise_description(description)
    if "DATE" in text:
        return "Date"
    if any(word in text for word in ("RATIO", "PERCENT", "PCT")):
        return "Percent or ratio as reported"
    if any(phrase in text for phrase in ("NUMBER OF", "NUM OF", "COUNT", "INDICATOR", "YES OR NO")):
        return "Count or indicator"
    return "USD thousands unless the applicable FFIEC instructions specify otherwise"


def find_column(columns, aliases):
    lookup = {str(column).strip().upper(): column for column in columns}
    for alias in aliases:
        match = lookup.get(alias.strip().upper())
        if match is not None:
            return match
    return None


def read_two_header_rows(path):
    encoding = detect_encoding(path)
    delimiter = detect_delimiter(path, encoding)
    with path.open("r", encoding=encoding, errors="replace") as file:
        header_line = file.readline()
        description_line = file.readline()

    columns = next(csv.reader([header_line], delimiter=delimiter))
    descriptions = next(csv.reader([description_line], delimiter=delimiter))
    columns = [str(column).strip().lstrip("\ufeff") for column in columns]
    descriptions = [str(description).strip() for description in descriptions]
    maximum_length = max(len(columns), len(descriptions))
    columns += [""] * (maximum_length - len(columns))
    descriptions += [""] * (maximum_length - len(descriptions))

    records = []
    for column_position, (column, description) in enumerate(zip(columns, descriptions), start=1):
        if not column:
            continue
        upper_column = column.upper()
        is_financial_code = bool(FINANCIAL_CODE_PATTERN.fullmatch(upper_column))
        records.append(
            {
                "column_position": column_position,
                "variable_code": upper_column,
                "original_column_name": column,
                "description": description,
                "is_financial_code": is_financial_code,
                "code_prefix": upper_column[:4] if is_financial_code else None,
                "item_code": upper_column[4:] if is_financial_code else None,
            }
        )
    return {
        "encoding": encoding,
        "delimiter": delimiter,
        "records": records,
        "columns": [record["original_column_name"] for record in records],
    }


def read_reporting_periods(path, header_result):
    date_column = find_column(header_result["columns"], DATE_ALIASES)
    if date_column is None:
        raise ValueError("Reporting-period column not found in first header row.")
    reporting_dates = set()
    reader = pd.read_csv(
        path,
        sep=header_result["delimiter"],
        header=0,
        skiprows=[1],
        usecols=[date_column],
        dtype=str,
        encoding=header_result["encoding"],
        chunksize=100_000,
        low_memory=False,
    )
    for chunk in reader:
        date_values = pd.to_datetime(chunk[date_column], errors="coerce").dropna()
        reporting_dates.update(date_values.dt.to_period("Q").astype(str))
    return sorted(reporting_dates, key=lambda value: pd.Period(value, freq="Q"))


print("FFIEC audit helpers created.")


## 5. Scan headers and reporting periods from file contents


In [ ]:
# Scan headers and reporting periods from file contents

file_inventory_records = []
variable_file_records = []
reporting_period_records = []
unreadable_records = []

for file_id, path in enumerate(tqdm(ffiec_files, desc="Scanning FFIEC files"), start=1):
    relative_path = path.relative_to(FFIEC_RAW_DIR)
    try:
        header_result = read_two_header_rows(path)
        reporting_periods = read_reporting_periods(path, header_result)
        financial_variables = [
            record for record in header_result["records"] if record["is_financial_code"]
        ]
        file_inventory_records.append(
            {
                "file_id": file_id,
                "relative_path": str(relative_path),
                "filename": path.name,
                "file_year_hint": extract_file_year(path),
                "encoding": header_result["encoding"],
                "delimiter": repr(header_result["delimiter"]),
                "total_columns": len(header_result["records"]),
                "financial_variable_count": len(financial_variables),
                "reporting_period_count": len(reporting_periods),
                "first_reporting_quarter": reporting_periods[0] if reporting_periods else None,
                "last_reporting_quarter": reporting_periods[-1] if reporting_periods else None,
                "reporting_quarters": ";".join(reporting_periods),
                "size_mb": round(path.stat().st_size / (1024 ** 2), 4),
                "readable": True,
                "read_error": None,
            }
        )
        for quarter in reporting_periods:
            reporting_period_records.append(
                {"file_id": file_id, "relative_path": str(relative_path), "quarter": quarter}
            )
        for record in header_result["records"]:
            variable_file_records.append(
                {
                    "file_id": file_id,
                    "relative_path": str(relative_path),
                    "filename": path.name,
                    "file_year_hint": extract_file_year(path),
                    "reporting_quarters": ";".join(reporting_periods),
                    **record,
                }
            )
    except Exception as error:
        unreadable_records.append(
            {
                "file_id": file_id,
                "relative_path": str(relative_path),
                "error": f"{type(error).__name__}: {error}",
            }
        )

file_inventory = pd.DataFrame(file_inventory_records)
variable_file_inventory = pd.DataFrame(variable_file_records)
reporting_period_inventory = pd.DataFrame(reporting_period_records)
unreadable_files = pd.DataFrame(
    unreadable_records,
    columns=["file_id", "relative_path", "error"],
)
unsupported_file_inventory = pd.DataFrame(
    [
        {
            "relative_path": str(path.relative_to(FFIEC_RAW_DIR)),
            "filename": path.name,
            "extension": path.suffix.lower(),
            "size_mb": round(path.stat().st_size / (1024 ** 2), 4),
            "status": "recorded_not_parsed_by_notebook_02",
        }
        for path in unsupported_ffiec_files
    ],
    columns=["relative_path", "filename", "extension", "size_mb", "status"],
)

if variable_file_inventory.empty:
    raise ValueError("No FFIEC variables were recovered from supported files.")

print("FFIEC header and calendar scan completed.")
print(f"Readable supported files: {len(file_inventory):,}")
print(f"Unreadable supported files: {len(unreadable_files):,}")
display(file_inventory.head(10))


## 6. Build historical variable and quarter availability tables


In [ ]:
# Build historical variable and quarter availability tables

financial_variable_files = variable_file_inventory.loc[
    variable_file_inventory["is_financial_code"].eq(True)
].copy()
if financial_variable_files.empty:
    raise ValueError("No valid eight-character FFIEC financial codes were detected.")

expected_quarters = pd.period_range(
    DATA_SCOPE["expected_ffiec_start"],
    DATA_SCOPE["expected_ffiec_end"],
    freq="Q",
)
expected_quarter_strings = expected_quarters.astype(str).tolist()
detected_quarters = sorted(
    reporting_period_inventory["quarter"].dropna().unique(),
    key=lambda value: pd.Period(value, freq="Q"),
)
missing_reporting_quarters = sorted(
    set(expected_quarter_strings) - set(detected_quarters),
    key=lambda value: pd.Period(value, freq="Q"),
)
unexpected_reporting_quarters = sorted(
    set(detected_quarters) - set(expected_quarter_strings),
    key=lambda value: pd.Period(value, freq="Q"),
)

availability_records = []
variable_quarter_records = []

for variable_code, group in tqdm(
    financial_variable_files.groupby("variable_code"),
    desc="Building variable history",
):
    quarter_values = set()
    for quarter_text in group["reporting_quarters"].dropna():
        quarter_values.update(value for value in str(quarter_text).split(";") if value)
    quarter_values = sorted(quarter_values, key=lambda value: pd.Period(value, freq="Q"))

    description_values = [
        value for value in group["description"].dropna().astype(str).unique() if value.strip()
    ]
    description_counts = Counter(group["description"].fillna("").astype(str))
    description = description_counts.most_common(1)[0][0] if description_counts else ""

    if quarter_values:
        first_quarter = quarter_values[0]
        last_quarter = quarter_values[-1]
        continuous_range = pd.period_range(first_quarter, last_quarter, freq="Q").astype(str).tolist()
        internal_missing = sorted(
            set(continuous_range) - set(quarter_values),
            key=lambda value: pd.Period(value, freq="Q"),
        )
    else:
        first_quarter = None
        last_quarter = None
        internal_missing = []

    code_prefix = group["code_prefix"].dropna().iloc[0]
    item_code = group["item_code"].dropna().iloc[0]
    years_available = sorted({int(value[:4]) for value in quarter_values})

    availability_records.append(
        {
            "variable_code": variable_code,
            "code_prefix": code_prefix,
            "item_code": item_code,
            "description": description,
            "normalised_description": normalise_description(description),
            "description_variant_count": len(description_values),
            "description_variants": " | ".join(description_values),
            "inferred_raw_unit": infer_raw_unit(description),
            "file_count": int(group["file_id"].nunique()),
            "quarter_count": len(quarter_values),
            "first_quarter": first_quarter,
            "last_quarter": last_quarter,
            "years_available": ";".join(map(str, years_available)),
            "quarters_available": ";".join(quarter_values),
            "internal_gap_count": len(internal_missing),
            "internal_missing_quarters": ";".join(internal_missing),
            "late_start_flag": (
                first_quarter is not None
                and pd.Period(first_quarter, freq="Q") > expected_quarters[0]
            ),
            "discontinued_flag": (
                last_quarter is not None
                and pd.Period(last_quarter, freq="Q") < expected_quarters[-1]
            ),
        }
    )
    for quarter in quarter_values:
        variable_quarter_records.append(
            {
                "variable_code": variable_code,
                "code_prefix": code_prefix,
                "item_code": item_code,
                "quarter": quarter,
                "year": int(quarter[:4]),
                "available_in_source_file": True,
            }
        )

variable_availability = pd.DataFrame(availability_records).sort_values("variable_code").reset_index(drop=True)
variable_quarter_availability = pd.DataFrame(variable_quarter_records).sort_values(
    ["variable_code", "quarter"]
).reset_index(drop=True)
discontinued_fields = variable_availability.loc[
    variable_availability["discontinued_flag"]
].copy().reset_index(drop=True)

print(f"Detected reporting calendar: {detected_quarters[0]} to {detected_quarters[-1]}")
print(f"Reporting quarters detected: {len(detected_quarters):,}")
print(f"Missing expected reporting quarters: {missing_reporting_quarters}")
print(f"Unique FFIEC financial variables: {len(variable_availability):,}")
print(f"Variables with internal gaps: {variable_availability['internal_gap_count'].gt(0).sum():,}")
print(f"Late-start variables: {variable_availability['late_start_flag'].sum():,}")
print(f"Discontinued variables: {variable_availability['discontinued_flag'].sum():,}")
display(variable_availability.head(20))


## 7. Identify RCFD/RCON equivalents and possible historical code changes


In [ ]:
# Identify RCFD/RCON equivalents and possible historical code changes

availability_lookup = variable_availability.set_index("variable_code").to_dict("index")
equivalence_records = []
for item_code in sorted(variable_availability["item_code"].dropna().unique()):
    rcfd_code = f"RCFD{item_code}"
    rcon_code = f"RCON{item_code}"
    if rcfd_code not in availability_lookup or rcon_code not in availability_lookup:
        continue
    rcfd = availability_lookup[rcfd_code]
    rcon = availability_lookup[rcon_code]
    description_match = (
        normalise_description(rcfd["description"])
        == normalise_description(rcon["description"])
    )
    equivalence_records.append(
        {
            "item_code": item_code,
            "rcfd_code": rcfd_code,
            "rcon_code": rcon_code,
            "rcfd_description": rcfd["description"],
            "rcon_description": rcon["description"],
            "description_match": description_match,
            "rcfd_first_quarter": rcfd["first_quarter"],
            "rcfd_last_quarter": rcfd["last_quarter"],
            "rcon_first_quarter": rcon["first_quarter"],
            "rcon_last_quarter": rcon["last_quarter"],
            "equivalence_status": (
                "AUTO_CONFIRMED_DESCRIPTION_MATCH"
                if description_match
                else "REVIEW_DESCRIPTION_MISMATCH"
            ),
            "mapping_rule": (
                "Prefer consolidated RCFD; use RCON only when the item definition is equivalent. "
                "Never sum RCFD and RCON."
            ),
        }
    )
rcfd_rcon_equivalence = pd.DataFrame(
    equivalence_records,
    columns=[
        "item_code", "rcfd_code", "rcon_code", "rcfd_description", "rcon_description",
        "description_match", "rcfd_first_quarter", "rcfd_last_quarter",
        "rcon_first_quarter", "rcon_last_quarter", "equivalence_status", "mapping_rule",
    ],
).sort_values("item_code").reset_index(drop=True)

code_change_records = []
for normalised_description, group in variable_availability.loc[
    variable_availability["normalised_description"].ne("")
].groupby("normalised_description"):
    codes = sorted(group["variable_code"].unique())
    if len(codes) <= 1:
        continue
    only_rcfd_rcon_pair = (
        group["item_code"].nunique() == 1
        and set(group["code_prefix"]).issubset({"RCFD", "RCON"})
    )
    if only_rcfd_rcon_pair:
        continue
    starts = group["first_quarter"].dropna().tolist()
    ends = group["last_quarter"].dropna().tolist()
    code_change_records.append(
        {
            "normalised_description": normalised_description,
            "representative_description": group["description"].iloc[0],
            "code_count": len(codes),
            "codes": ";".join(codes),
            "first_quarter": min(starts, key=lambda value: pd.Period(value, freq="Q")) if starts else None,
            "last_quarter": max(ends, key=lambda value: pd.Period(value, freq="Q")) if ends else None,
            "review_status": "REVIEW_POSSIBLE_REPLACEMENT_OR_DUPLICATE_CONCEPT",
            "review_reason": "Same normalised description appears under multiple codes beyond a simple RCFD/RCON pair.",
        }
    )
possible_code_changes = pd.DataFrame(
    code_change_records,
    columns=[
        "normalised_description", "representative_description", "code_count", "codes",
        "first_quarter", "last_quarter", "review_status", "review_reason",
    ],
).sort_values(["code_count", "representative_description"], ascending=[False, True]).reset_index(drop=True)

print(f"RCFD/RCON equivalent pairs: {len(rcfd_rcon_equivalence):,}")
print(f"Description-mismatch pairs requiring review: {(~rcfd_rcon_equivalence['description_match']).sum():,}")
print(f"Possible historical code-change groups: {len(possible_code_changes):,}")
display(rcfd_rcon_equivalence.head(20))
display(possible_code_changes.head(20))


## 8. Confirm every candidate predictor’s definition, unit and sign treatment


In [ ]:
# Confirm every candidate predictor’s definition, unit and sign treatment

def candidate(camels_area, predictor, definition, unit, construction, risk_direction,
              source_concepts, status="Main candidate", sign_treatment="Retain source sign; do not take absolute values or clip genuine negatives"):
    return {
        "camels_area": camels_area,
        "predictor": predictor,
        "definition": definition,
        "unit": unit,
        "construction": construction,
        "risk_direction": risk_direction,
        "sign_treatment": sign_treatment,
        "source_concepts": source_concepts,
        "main_period_status": status,
    }

candidate_predictor_records = [
    candidate("Size", "log_total_assets", "Natural logarithm of total assets.", "Natural log of USD thousands", "ln(total assets)", "Mixed; size may increase complexity but also diversification", "total_assets", sign_treatment="Total assets must be positive before taking the natural log"),
    candidate("Capital", "equity_to_assets", "Total equity capital divided by total assets.", "Ratio", "total equity capital / total assets", "Lower values indicate greater risk", "total_equity_capital;total_assets", sign_treatment="Retain reported equity sign; negative equity is economically meaningful"),
    candidate("Capital", "tier1_leverage_ratio", "Reported Tier 1 leverage capital ratio.", "Percent or ratio, normalised consistently after code confirmation", "reported regulatory ratio", "Lower values indicate greater risk", "tier1_leverage_capital", "Subject to historical consistency"),
    candidate("Capital", "risk_based_capital_ratio", "Historically consistent reported risk-based capital ratio.", "Percent or ratio, normalised consistently after code confirmation", "reported regulatory ratio", "Lower values indicate greater risk", "risk_based_capital", "Subject to Basel reporting consistency"),
    candidate("Asset quality", "noncurrent_loans_to_loans", "Loans 90 days or more past due plus nonaccrual loans, divided by total loans.", "Ratio", "(90+ days past due loans + nonaccrual loans) / total loans", "Higher values indicate greater risk", "past_due_90_days;nonaccrual_loans;total_loans", sign_treatment="Components are expected non-negative; investigate rather than silently clip exceptions"),
    candidate("Asset quality", "allowance_to_loans", "Allowance for credit losses divided by total loans.", "Ratio", "allowance for credit losses / total loans", "Mixed; protection versus deteriorating credit quality", "allowance_credit_losses;total_loans"),
    candidate("Asset quality", "provisions_to_average_assets", "Trailing-four-quarter credit-loss provisions divided by average assets.", "Ratio", "TTM provisions / average assets", "Higher values generally indicate greater risk", "provision_credit_losses;total_assets", sign_treatment="Retain genuine negative provisions or recoveries"),
    candidate("Asset quality", "net_chargeoffs_to_loans", "Trailing-four-quarter net charge-offs divided by average loans.", "Ratio", "TTM net charge-offs / average loans", "Higher values indicate greater risk", "net_chargeoffs;total_loans", "Subject to historical availability", "Retain genuine negative net charge-offs or recoveries"),
    candidate("Asset quality", "oreo_to_assets", "Other real estate owned divided by total assets.", "Ratio", "other real estate owned / total assets", "Higher values indicate greater risk", "other_real_estate_owned;total_assets", sign_treatment="OREO is expected non-negative; investigate rather than silently clip exceptions"),
    candidate("Earnings", "return_on_average_assets", "Trailing-four-quarter net income divided by average assets.", "Ratio", "TTM net income / average assets", "Lower values indicate greater risk", "net_income;total_assets", sign_treatment="Retain negative income and negative profitability"),
    candidate("Earnings", "net_interest_margin", "Trailing-four-quarter net interest income divided by average earning assets or average assets where consistently defined.", "Ratio", "TTM net interest income / appropriate average denominator", "Lower values generally indicate greater risk", "net_interest_income;total_assets"),
    candidate("Earnings", "interest_expense_to_average_assets", "Trailing-four-quarter interest expense divided by average assets.", "Ratio", "TTM interest expense / average assets", "Higher values generally indicate greater funding pressure", "total_interest_expense;total_assets"),
    candidate("Earnings", "noninterest_income_to_average_assets", "Trailing-four-quarter noninterest income divided by average assets.", "Ratio", "TTM noninterest income / average assets", "Mixed", "total_noninterest_income;total_assets", sign_treatment="Retain genuine negative noninterest income"),
    candidate("Management proxy", "efficiency_ratio", "Noninterest expense divided by net interest income plus noninterest income.", "Ratio", "TTM noninterest expense / (TTM net interest income + TTM noninterest income)", "Higher values indicate lower operating efficiency", "total_noninterest_expense;net_interest_income;total_noninterest_income", sign_treatment="Retain source signs; set undefined or zero-denominator ratios to missing"),
    candidate("Management proxy", "annual_asset_growth", "Percentage change in total assets from the same quarter one year earlier.", "Annual growth rate", "(assets_t / assets_t-4) - 1", "Unusually high growth may indicate greater risk", "total_assets"),
    candidate("Liquidity and funding", "loans_to_assets", "Total loans divided by total assets.", "Ratio", "total loans / total assets", "Higher values may indicate lower balance-sheet liquidity", "total_loans;total_assets"),
    candidate("Liquidity and funding", "loans_to_deposits", "Total loans divided by total deposits.", "Ratio", "total loans / total deposits", "Higher values generally indicate greater liquidity pressure", "total_loans;total_deposits", sign_treatment="Set zero-denominator ratios to missing; do not replace with zero"),
    candidate("Liquidity and funding", "deposits_to_assets", "Total deposits divided by total assets.", "Ratio", "total deposits / total assets", "Lower values may indicate reliance on non-deposit funding", "total_deposits;total_assets"),
    candidate("Liquidity and funding", "liquid_assets_to_assets", "Consistently available liquid assets divided by total assets.", "Ratio", "liquid assets / total assets", "Lower values indicate lower liquidity", "cash_and_balances;fed_funds_sold;total_assets", "Subject to component consistency"),
    candidate("Liquidity and funding", "brokered_deposits_ratio", "Brokered deposits divided by total deposits.", "Ratio", "brokered deposits / total deposits", "Higher values generally indicate greater funding risk", "brokered_deposits;total_deposits", "Subject to historical availability"),
    candidate("Portfolio exposure", "securities_to_assets", "Consistently reported securities holdings divided by total assets.", "Ratio", "securities / total assets", "Mixed", "held_to_maturity_securities;available_for_sale_securities;trading_assets;total_assets", "Subject to component consistency"),
    candidate("Portfolio exposure", "commercial_real_estate_concentration", "Commercial real-estate loans divided by total loans or capital using a consistently defined denominator.", "Ratio", "commercial real-estate loans / selected denominator", "Higher values indicate greater concentration risk", "multifamily_real_estate_loans;nonfarm_nonresidential_loans;total_loans", "Subject to historical consistency"),
    candidate("Portfolio exposure", "construction_development_concentration", "Construction and land-development loans divided by total loans or capital.", "Ratio", "construction and development loans / selected denominator", "Higher values indicate greater concentration risk", "construction_development_loans;total_loans", "Subject to historical consistency"),
    candidate("Growth", "annual_loan_growth", "Percentage change in total loans from the same quarter one year earlier.", "Annual growth rate", "(loans_t / loans_t-4) - 1", "Unusually high growth may indicate greater risk", "total_loans"),
    candidate("Growth", "annual_deposit_growth", "Percentage change in total deposits from the same quarter one year earlier.", "Annual growth rate", "(deposits_t / deposits_t-4) - 1", "Large declines may indicate funding stress", "total_deposits"),
    candidate("Securities risk", "unrealised_securities_gain_loss_ratio", "Unrealised securities gains or losses divided by assets or capital.", "Ratio", "unrealised securities gains or losses / assets or capital", "More negative values indicate greater securities risk", "unrealised_securities_gain_loss;accumulated_other_comprehensive_income;total_assets;total_equity_capital", "Use only if historically consistent codes are confirmed", "Retain positive gains and negative losses exactly as reported"),
    candidate("Supplementary funding risk", "uninsured_deposits_ratio", "Estimated uninsured deposits divided by total deposits.", "Ratio", "estimated uninsured deposits / total deposits", "Higher values indicate greater run risk", "uninsured_deposits;total_deposits", "Recent-period supplementary diagnostic only"),
]
candidate_predictor_register = pd.DataFrame(candidate_predictor_records)

required_definition_fields = (
    "predictor", "definition", "unit", "construction", "risk_direction", "sign_treatment", "source_concepts"
)
assert candidate_predictor_register["predictor"].is_unique
assert not candidate_predictor_register[list(required_definition_fields)].isna().any().any()
assert not candidate_predictor_register[list(required_definition_fields)].eq("").any().any()

print(f"Candidate predictors fully documented: {len(candidate_predictor_register):,}")
print("Definition, unit, construction, risk direction and sign treatment are complete for every row.")
display(candidate_predictor_register)


## 9. Match candidate source concepts to observed FFIEC codes


In [ ]:
# Match candidate source concepts to observed FFIEC codes

candidate_concept_patterns = {
    "total_assets": r"^TOTAL ASSETS$",
    "total_equity_capital": r"^TOTAL EQUITY CAPITAL$",
    "tier1_leverage_capital": r"TIER 1.*LEVERAGE",
    "risk_based_capital": r"(?:TIER 1|TOTAL).*RISK.*CAPITAL",
    "total_loans": r"LOANS.*LEASES.*NET(?: OF)? UNEARNED INCOME$",
    "total_deposits": r"^TOTAL DEPOSITS$",
    "past_due_90_days": r"(?:90 DAYS|90DAYS|>=90|GE 90).*PAST DUE|PAST DUE.*(?:90 DAYS|90DAYS|>=90|GE 90)",
    "nonaccrual_loans": r"(?:TOTAL.*NONACCRUAL|NONACCRUAL.*TOTAL)",
    "allowance_credit_losses": r"(?:ALLOWANCE.*CREDIT LOSSES|ALLOWANCE.*LOAN.*LEASE|^ALLL AMT$)",
    "provision_credit_losses": r"^PROVISION FOR (?:CREDIT|LOAN).*LOSSES$",
    "net_chargeoffs": r"NET CHARGE.?OFF",
    "other_real_estate_owned": r"^OTHER REAL ESTATE OWNED$",
    "net_income": r"^NET INCOME$",
    "net_interest_income": r"^NET INTEREST INCOME$",
    "total_interest_expense": r"^TOTAL INTEREST EXPENSE$",
    "total_interest_income": r"^TOTAL INTEREST INCOME$",
    "total_noninterest_income": r"^TOTAL NONINTEREST INCOME$",
    "total_noninterest_expense": r"^TOTAL NONINTEREST EXPENSE$",
    "cash_and_balances": r"^CASH AND BALANCES DUE",
    "fed_funds_sold": r"^FED.*FUNDS SOLD",
    "held_to_maturity_securities": r"^HELD.TO.MATURITY SECURITIES$",
    "available_for_sale_securities": r"^AVAILABLE.FOR.SALE SECURITIES$",
    "trading_assets": r"^TOTAL TRADING ASSETS$",
    "brokered_deposits": r"BROKERED DEPOSITS",
    "construction_development_loans": r"CONSTRUCTION.*LAND DEVELOPMENT.*LOANS",
    "multifamily_real_estate_loans": r"LOANS.*MULTIFAMILY",
    "nonfarm_nonresidential_loans": r"LOANS.*NONFARM.*NONRESIDENTIAL",
    "accumulated_other_comprehensive_income": r"ACCUMULATED.*COMPREHENSIVE INCOME",
    "unrealised_securities_gain_loss": r"UNREALI[ZS]ED.*(?:GAIN|LOSS).*SECUR",
    "uninsured_deposits": r"UNINSURED DEPOSIT",
}

candidate_match_records = []
for concept, pattern in candidate_concept_patterns.items():
    matches = variable_availability.loc[
        variable_availability["normalised_description"].str.contains(
            pattern, case=False, regex=True, na=False
        )
    ]
    for _, row in matches.iterrows():
        candidate_match_records.append(
            {
                "candidate_concept": concept,
                "variable_code": row["variable_code"],
                "code_prefix": row["code_prefix"],
                "item_code": row["item_code"],
                "description": row["description"],
                "inferred_raw_unit": row["inferred_raw_unit"],
                "first_quarter": row["first_quarter"],
                "last_quarter": row["last_quarter"],
                "quarter_count": row["quarter_count"],
                "internal_gap_count": row["internal_gap_count"],
                "late_start_flag": row["late_start_flag"],
                "discontinued_flag": row["discontinued_flag"],
                "mapping_status": "AUTO_DESCRIPTION_MATCH_REVIEW_BEFORE_EXTRACTION",
            }
        )

candidate_concept_matches = pd.DataFrame(
    candidate_match_records,
    columns=[
        "candidate_concept", "variable_code", "code_prefix", "item_code", "description",
        "inferred_raw_unit", "first_quarter", "last_quarter", "quarter_count",
        "internal_gap_count", "late_start_flag", "discontinued_flag", "mapping_status",
    ],
).sort_values(["candidate_concept", "first_quarter", "variable_code"]).reset_index(drop=True)

mapping_summary_records = []
for concept in candidate_concept_patterns:
    group = candidate_concept_matches.loc[
        candidate_concept_matches["candidate_concept"].eq(concept)
    ]
    code_count = int(group["variable_code"].nunique())
    if code_count == 0:
        mapping_status = "MISSING_NO_AUTOMATIC_MATCH"
    elif code_count == 1:
        mapping_status = "SINGLE_DESCRIPTION_MATCH_CONFIRM_IN_NOTEBOOK_03"
    else:
        mapping_status = "MULTIPLE_HISTORICAL_CODES_RECONCILE_IN_NOTEBOOK_03"
    mapping_summary_records.append(
        {
            "candidate_concept": concept,
            "code_count": code_count,
            "codes": ";".join(sorted(group["variable_code"].unique())),
            "prefixes": ";".join(sorted(group["code_prefix"].dropna().unique())),
            "first_quarter": (
                min(group["first_quarter"].dropna(), key=lambda value: pd.Period(value, freq="Q"))
                if group["first_quarter"].notna().any() else None
            ),
            "last_quarter": (
                max(group["last_quarter"].dropna(), key=lambda value: pd.Period(value, freq="Q"))
                if group["last_quarter"].notna().any() else None
            ),
            "mapping_status": mapping_status,
        }
    )
candidate_concept_mapping_summary = pd.DataFrame(mapping_summary_records)

matched_concepts = set(candidate_concept_matches["candidate_concept"])
unmatched_concepts = sorted(set(candidate_concept_patterns) - matched_concepts)
required_core_concepts = {
    "total_assets", "total_equity_capital", "total_loans", "total_deposits",
    "provision_credit_losses", "other_real_estate_owned", "net_income",
    "net_interest_income", "total_interest_expense", "total_noninterest_income",
    "total_noninterest_expense",
}
missing_core_concepts = sorted(required_core_concepts - matched_concepts)

print(f"Candidate source-code matches: {len(candidate_concept_matches):,}")
print(f"Concepts without automatic matches: {unmatched_concepts}")
print(f"Missing required core concepts: {missing_core_concepts}")
display(candidate_concept_mapping_summary)
display(candidate_concept_matches.head(50))


## 10. Profile FFIEC 031/041/051 non-missingness for candidate codes


In [ ]:
# Profile FFIEC 031/041/051 non-missingness for candidate codes

candidate_codes = sorted(candidate_concept_matches["variable_code"].unique())
structural_profile_records = []
structural_skip_records = []
structurally_profiled_files = 0

print("Profiling candidate fields by filing form. This is the longest Notebook 02 step.")

for path in tqdm(ffiec_files, desc="Profiling FFIEC forms"):
    try:
        header_result = read_two_header_rows(path)
        actual_columns = header_result["columns"]
        code_lookup = {
            record["variable_code"]: record["original_column_name"]
            for record in header_result["records"] if record["is_financial_code"]
        }
        present_codes = [code for code in candidate_codes if code in code_lookup]
        if not present_codes:
            structural_skip_records.append(
                {"relative_path": str(path.relative_to(FFIEC_RAW_DIR)), "reason": "no_candidate_codes"}
            )
            continue

        date_column = find_column(actual_columns, DATE_ALIASES)
        form_column = find_column(actual_columns, FORM_ALIASES)
        if date_column is None or form_column is None:
            structural_skip_records.append(
                {
                    "relative_path": str(path.relative_to(FFIEC_RAW_DIR)),
                    "reason": "date_or_filing_form_column_not_found",
                }
            )
            continue

        original_candidate_columns = [code_lookup[code] for code in present_codes]
        use_columns = [date_column, form_column, *original_candidate_columns]
        reverse_code_lookup = {code_lookup[code]: code for code in present_codes}

        reader = pd.read_csv(
            path,
            sep=header_result["delimiter"],
            header=0,
            skiprows=[1],
            usecols=use_columns,
            dtype=str,
            encoding=header_result["encoding"],
            chunksize=50_000,
            low_memory=False,
            na_values=["", " ", "NA", "N/A", "NULL"],
            keep_default_na=True,
        )
        file_had_usable_rows = False
        for chunk in reader:
            chunk["quarter"] = pd.to_datetime(
                chunk[date_column], errors="coerce"
            ).dt.to_period("Q").astype(str)
            chunk["filing_form"] = (
                chunk[form_column].astype("string").str.strip().str.extract(r"(031|041|051)", expand=False)
            )
            chunk = chunk.loc[
                chunk["quarter"].ne("NaT") & chunk["filing_form"].notna()
            ].copy()
            if chunk.empty:
                continue
            file_had_usable_rows = True
            for original_name in original_candidate_columns:
                chunk[original_name] = chunk[original_name].replace(r"^\s*$", np.nan, regex=True)

            group_columns = ["quarter", "filing_form"]
            row_counts = chunk.groupby(group_columns).size().rename("rows").reset_index()
            nonmissing_counts = (
                chunk.groupby(group_columns)[original_candidate_columns]
                .count().reset_index()
                .melt(id_vars=group_columns, var_name="original_column", value_name="nonmissing")
            )
            profile_chunk = nonmissing_counts.merge(row_counts, on=group_columns, how="left")
            profile_chunk["variable_code"] = profile_chunk["original_column"].map(reverse_code_lookup)
            structural_profile_records.extend(
                profile_chunk[["quarter", "filing_form", "variable_code", "rows", "nonmissing"]]
                .to_dict("records")
            )
        if file_had_usable_rows:
            structurally_profiled_files += 1
        else:
            structural_skip_records.append(
                {"relative_path": str(path.relative_to(FFIEC_RAW_DIR)), "reason": "no_parseable_form_rows"}
            )
    except Exception as error:
        structural_skip_records.append(
            {
                "relative_path": str(path.relative_to(FFIEC_RAW_DIR)),
                "reason": f"{type(error).__name__}: {error}",
            }
        )

structural_profile = pd.DataFrame(
    structural_profile_records,
    columns=["quarter", "filing_form", "variable_code", "rows", "nonmissing"],
)
structural_profile_skipped_files = pd.DataFrame(
    structural_skip_records,
    columns=["relative_path", "reason"],
)

if not structural_profile.empty:
    structural_profile = structural_profile.groupby(
        ["quarter", "filing_form", "variable_code"], as_index=False
    )[["rows", "nonmissing"]].sum()
    structural_profile["nonmissing_rate"] = (
        structural_profile["nonmissing"] / structural_profile["rows"]
    )

profile_051 = structural_profile.loc[
    structural_profile["filing_form"].eq("051")
].rename(
    columns={
        "rows": "rows_051",
        "nonmissing": "nonmissing_051",
        "nonmissing_rate": "nonmissing_rate_051",
    }
)
profile_other = (
    structural_profile.loc[~structural_profile["filing_form"].eq("051")]
    .groupby(["quarter", "variable_code"], as_index=False)[["rows", "nonmissing"]].sum()
    .rename(columns={"rows": "rows_other_forms", "nonmissing": "nonmissing_other_forms"})
)
if not profile_other.empty:
    profile_other["nonmissing_rate_other_forms"] = (
        profile_other["nonmissing_other_forms"] / profile_other["rows_other_forms"]
    )

structural_comparison = profile_051[
    ["quarter", "variable_code", "rows_051", "nonmissing_051", "nonmissing_rate_051"]
].merge(profile_other, on=["quarter", "variable_code"], how="outer")
structural_comparison["ffiec_051_structural_flag"] = (
    structural_comparison["rows_051"].fillna(0).ge(25)
    & structural_comparison["rows_other_forms"].fillna(0).ge(25)
    & structural_comparison["nonmissing_rate_051"].fillna(0).lt(0.05)
    & structural_comparison["nonmissing_rate_other_forms"].fillna(0).gt(0.25)
)
ffiec_051_structural_flags = (
    structural_comparison.loc[structural_comparison["ffiec_051_structural_flag"]]
    .merge(variable_availability[["variable_code", "description"]], on="variable_code", how="left")
    .sort_values(["variable_code", "quarter"])
    .reset_index(drop=True)
)

print(f"Files contributing to filing-form profile: {structurally_profiled_files:,}")
print(f"Candidate variable-quarter-form rows: {len(structural_profile):,}")
print(f"FFIEC 051 structural non-reporting flags: {len(ffiec_051_structural_flags):,}")
display(ffiec_051_structural_flags.head(30))


## 11. Screen Basel-era capital reporting changes and separate uninsured deposits


In [ ]:
# Screen Basel-era capital reporting changes and separate uninsured deposits

capital_concepts = {"tier1_leverage_capital", "risk_based_capital", "total_equity_capital"}
capital_codes = sorted(
    candidate_concept_matches.loc[
        candidate_concept_matches["candidate_concept"].isin(capital_concepts),
        "variable_code",
    ].unique()
)
basel_break_records = []
for variable_code in capital_codes:
    history = structural_profile.loc[
        structural_profile["variable_code"].eq(variable_code)
    ].copy()
    history["period"] = pd.PeriodIndex(history["quarter"], freq="Q")
    pre_period = history.loc[
        history["period"].between(pd.Period("2011Q1"), pd.Period("2013Q4"))
    ]
    post_period = history.loc[
        history["period"].between(pd.Period("2015Q1"), pd.Period("2017Q4"))
    ]
    pre_rate = (
        pre_period["nonmissing"].sum() / pre_period["rows"].sum()
        if pre_period["rows"].sum() > 0 else np.nan
    )
    post_rate = (
        post_period["nonmissing"].sum() / post_period["rows"].sum()
        if post_period["rows"].sum() > 0 else np.nan
    )
    rate_change = post_rate - pre_rate if pd.notna(pre_rate) and pd.notna(post_rate) else np.nan
    availability = availability_lookup.get(variable_code, {})
    basel_break_records.append(
        {
            "variable_code": variable_code,
            "description": availability.get("description"),
            "first_quarter": availability.get("first_quarter"),
            "last_quarter": availability.get("last_quarter"),
            "pre_2014_nonmissing_rate": pre_rate,
            "post_2014_nonmissing_rate": post_rate,
            "rate_change": rate_change,
            "basel_break_review_flag": (
                pd.isna(pre_rate) or pd.isna(post_rate)
                or (pd.notna(rate_change) and abs(rate_change) >= 0.25)
            ),
            "interpretation_limit": (
                "Screening flag around the 2014 Basel III reporting transition; "
                "not proof that Basel III caused the change."
            ),
        }
    )
basel_break_screen = pd.DataFrame(
    basel_break_records,
    columns=[
        "variable_code", "description", "first_quarter", "last_quarter",
        "pre_2014_nonmissing_rate", "post_2014_nonmissing_rate", "rate_change",
        "basel_break_review_flag", "interpretation_limit",
    ],
).sort_values("variable_code").reset_index(drop=True)

uninsured_deposit_inventory = candidate_concept_matches.loc[
    candidate_concept_matches["candidate_concept"].eq("uninsured_deposits")
].copy().reset_index(drop=True)
if not uninsured_deposit_inventory.empty:
    uninsured_deposit_inventory["usage_rule"] = (
        "Exclude from full-period main models; retain only for supplementary recent-period analysis."
    )
    uninsured_deposit_inventory["reporting_limitation"] = (
        "Availability is limited by reporting period and institution-size thresholds; "
        "do not interpret structural absence as zero."
    )

print(f"Capital fields screened: {len(basel_break_screen):,}")
print(f"Capital fields requiring Basel-era review: {basel_break_screen['basel_break_review_flag'].sum():,}")
print(f"Uninsured-deposit source fields recorded separately: {len(uninsured_deposit_inventory):,}")
display(basel_break_screen)
display(uninsured_deposit_inventory)


## 12. Build the audit summary and save all V2 outputs


In [ ]:
# Build the audit summary and save all V2 outputs

audit_rows = [
    {"check": "Notebook 01 V2 prerequisite", "status": "PASS", "detail": "Notebook 01 V2 completed with PASS and exact policy keys."},
    {"check": "V2 output isolation", "status": "PASS", "detail": str(NB02_OUTPUT_DIR)},
    {
        "check": "Supported FFIEC raw files readable",
        "status": "PASS" if unreadable_files.empty else "FAIL",
        "detail": f"{len(unreadable_files)} unreadable supported file(s)",
    },
    {
        "check": "Reporting-quarter coverage from file contents",
        "status": "PASS" if not missing_reporting_quarters else "FAIL",
        "detail": (
            f"All {len(expected_quarters)} expected quarters detected"
            if not missing_reporting_quarters else f"Missing quarters: {missing_reporting_quarters}"
        ),
    },
    {
        "check": "Unexpected out-of-scope reporting quarters",
        "status": "PASS" if not unexpected_reporting_quarters else "WARNING",
        "detail": f"Unexpected quarters: {unexpected_reporting_quarters}",
    },
    {
        "check": "Required core source concepts",
        "status": "PASS" if not missing_core_concepts else "FAIL",
        "detail": "All required core concepts matched" if not missing_core_concepts else f"Missing: {missing_core_concepts}",
    },
    {
        "check": "Candidate predictor definition/unit/sign register",
        "status": "PASS",
        "detail": f"{len(candidate_predictor_register)} predictors fully documented",
    },
    {
        "check": "RCFD/RCON equivalence inventory",
        "status": "PASS" if not rcfd_rcon_equivalence.empty else "WARNING",
        "detail": f"{len(rcfd_rcon_equivalence)} equivalent pairs recorded",
    },
    {
        "check": "Historical code-change and discontinued-field inventory",
        "status": "PASS",
        "detail": f"{len(possible_code_changes)} possible code-change groups; {len(discontinued_fields)} discontinued fields",
    },
    {
        "check": "FFIEC filing-form structural profile",
        "status": "PASS" if structurally_profiled_files > 0 else "FAIL",
        "detail": f"{structurally_profiled_files} files contributed; {len(ffiec_051_structural_flags)} FFIEC 051 flags",
    },
    {
        "check": "Basel-era capital screen",
        "status": "PASS" if not basel_break_screen.empty else "WARNING",
        "detail": f"{len(basel_break_screen)} capital fields screened; {basel_break_screen['basel_break_review_flag'].sum()} review flags",
    },
    {
        "check": "Uninsured-deposit fields separated",
        "status": "PASS" if not uninsured_deposit_inventory.empty else "WARNING",
        "detail": f"{len(uninsured_deposit_inventory)} fields recorded outside full-period main candidates",
    },
    {
        "check": "Unmatched non-core candidate concepts",
        "status": "PASS" if not unmatched_concepts else "WARNING",
        "detail": f"Unmatched concepts: {unmatched_concepts}",
    },
]
audit_summary = pd.DataFrame(audit_rows)
overall_status = "PASS" if not audit_summary["status"].eq("FAIL").any() else "REVIEW REQUIRED"

output_tables = {
    "ffiec_file_inventory_v2.csv": file_inventory,
    "ffiec_unsupported_file_inventory_v2.csv": unsupported_file_inventory,
    "ffiec_reporting_period_inventory_v2.csv": reporting_period_inventory,
    "ffiec_variable_file_inventory_v2.csv": variable_file_inventory,
    "ffiec_variable_historical_availability_v2.csv": variable_availability,
    "ffiec_variable_quarter_availability_v2.csv": variable_quarter_availability,
    "ffiec_discontinued_fields_v2.csv": discontinued_fields,
    "rcfd_rcon_equivalence_map_v2.csv": rcfd_rcon_equivalence,
    "possible_historical_code_changes_v2.csv": possible_code_changes,
    "candidate_predictor_definition_register_v2.csv": candidate_predictor_register,
    "candidate_concept_code_matches_v2.csv": candidate_concept_matches,
    "candidate_concept_mapping_summary_v2.csv": candidate_concept_mapping_summary,
    "ffiec_form_nonmissing_profile_v2.csv": structural_profile,
    "ffiec_051_structural_nonreporting_flags_v2.csv": ffiec_051_structural_flags,
    "ffiec_structural_profile_skipped_files_v2.csv": structural_profile_skipped_files,
    "basel_era_capital_break_screen_v2.csv": basel_break_screen,
    "uninsured_deposit_field_inventory_v2.csv": uninsured_deposit_inventory,
    "unreadable_files_v2.csv": unreadable_files,
    "audit_summary_v2.csv": audit_summary,
}
for filename, table in output_tables.items():
    output_path = NB02_OUTPUT_DIR / filename
    assert output_path.is_relative_to(V2_ROOT)
    table.to_csv(output_path, index=False)

audit_report = {
    "notebook": "02_FFIEC_Variable_Inventory_and_Code_Mapping_V2",
    "analysis_version": VERSION_2_MAIN_TIMING_POLICY["analysis_version"],
    "run_time_utc": datetime.now(timezone.utc).isoformat(),
    "overall_status": overall_status,
    "ffiec_supported_files_scanned": int(len(file_inventory)),
    "ffiec_unsupported_files_recorded": int(len(unsupported_file_inventory)),
    "reporting_quarters_detected": int(len(detected_quarters)),
    "first_reporting_quarter": detected_quarters[0] if detected_quarters else None,
    "last_reporting_quarter": detected_quarters[-1] if detected_quarters else None,
    "missing_reporting_quarters": missing_reporting_quarters,
    "unexpected_reporting_quarters": unexpected_reporting_quarters,
    "unique_financial_variables": int(len(variable_availability)),
    "discontinued_fields": int(len(discontinued_fields)),
    "rcfd_rcon_equivalent_pairs": int(len(rcfd_rcon_equivalence)),
    "possible_code_change_groups": int(len(possible_code_changes)),
    "candidate_predictors_documented": int(len(candidate_predictor_register)),
    "candidate_code_matches": int(len(candidate_concept_matches)),
    "missing_core_concepts": missing_core_concepts,
    "unmatched_candidate_concepts": unmatched_concepts,
    "files_contributing_to_form_profile": int(structurally_profiled_files),
    "ffiec_051_flags": int(len(ffiec_051_structural_flags)),
    "basel_review_fields": int(basel_break_screen["basel_break_review_flag"].sum()),
    "uninsured_deposit_fields": int(len(uninsured_deposit_inventory)),
    "output_folder": str(NB02_OUTPUT_DIR),
}
audit_json_path = NB02_OUTPUT_DIR / "audit_report_v2.json"
audit_json_path.write_text(json.dumps(audit_report, indent=2) + "\n", encoding="utf-8")

audit_table_lines = ["| Check | Status | Detail |", "|---|---|---|"]
for row in audit_summary.itertuples(index=False):
    safe_check = str(row.check).replace("|", "\\|")
    safe_status = str(row.status).replace("|", "\\|")
    safe_detail = str(row.detail).replace("|", "\\|").replace("\n", " ")
    audit_table_lines.append(f"| {safe_check} | {safe_status} | {safe_detail} |")
audit_markdown_path = NB02_OUTPUT_DIR / "audit_report_v2.md"
audit_markdown_path.write_text(
    "\n".join(
        [
            "# Notebook 02 V2 Audit Report", "",
            f"- Overall status: **{overall_status}**",
            f"- Reporting period: {audit_report['first_reporting_quarter']} to {audit_report['last_reporting_quarter']}",
            f"- Unique financial variables: {len(variable_availability):,}",
            f"- Candidate predictors documented: {len(candidate_predictor_register):,}",
            "", "## Checks", "", *audit_table_lines, "",
        ]
    ),
    encoding="utf-8",
)

display(audit_summary)
print(f"\nOVERALL NOTEBOOK 02 V2 STATUS: {overall_status}")
print(f"Outputs saved under:\n{NB02_OUTPUT_DIR}")


## 13. Final execution gate


In [ ]:
# Final execution gate

print("=" * 78)
print("NOTEBOOK 02 V2 — FINAL INVENTORY RESULT")
print("=" * 78)
print(f"Overall status: {overall_status}")
print(f"Supported FFIEC files scanned: {len(file_inventory):,}")
print(f"Reporting quarters detected: {len(detected_quarters):,}")
print(f"Reporting period: {detected_quarters[0]} to {detected_quarters[-1]}")
print(f"Missing reporting quarters: {missing_reporting_quarters}")
print(f"Unique financial variables: {len(variable_availability):,}")
print(f"Discontinued fields: {len(discontinued_fields):,}")
print(f"RCFD/RCON equivalent pairs: {len(rcfd_rcon_equivalence):,}")
print(f"Possible code-change groups: {len(possible_code_changes):,}")
print(f"Candidate predictors documented: {len(candidate_predictor_register):,}")
print(f"Candidate source-code matches: {len(candidate_concept_matches):,}")
print(f"Missing required core concepts: {missing_core_concepts}")
print(f"Files contributing to form profile: {structurally_profiled_files:,}")
print(f"FFIEC 051 structural flags: {len(ffiec_051_structural_flags):,}")
print(f"Basel-era capital review fields: {basel_break_screen['basel_break_review_flag'].sum():,}")
print(f"Uninsured-deposit fields: {len(uninsured_deposit_inventory):,}")
print(f"Output folder: {NB02_OUTPUT_DIR}")

expected_output_paths = [NB02_OUTPUT_DIR / filename for filename in output_tables]
expected_output_paths.extend([audit_json_path, audit_markdown_path])
assert all(path.exists() for path in expected_output_paths), "One or more Notebook 02 V2 outputs are missing."
assert not audit_summary["status"].eq("FAIL").any(), (
    "Notebook 02 V2 contains FAIL results. Review audit_summary_v2.csv before Notebook 03."
)

if audit_summary["status"].eq("WARNING").any():
    print("\nNotebook 02 completed with warnings. Review the final results and warning rows.")
else:
    print("\nNotebook 02 completed successfully with no warnings.")
print("Do not run Notebook 03 until the Notebook 02 audit has been reviewed.")
